# ExaHyPE 2 Tutorials, Part 1: The Acoustic Equations

This is the first part of the ExaHyPE 2 tutorials.
These are defined within the Peano 4 documentation, which you can find online at https://tum-i5.github.io/ExaHyPE-Documentation/ or generate locally using `doxygen documentation/Doxyfile`.

This part of the tutorial will briefly go over a few things:
- How to import the ExaHyPE 2 API in Python
- The structure of an ExaHyPE 2 configuration file
- How to define a PDE solver in ExaHyPE 2
- Running and visualising the results of an ExaHyPE 2 implementation

This file is accompanied by a "sister file" called **PlanarAcousticSolver.cpp** which contains the C++ implementation of the solver as defined here.

The first thing we will do is import the ExaHyPE 2 API. Python needs to be able to find the Peano 4 and ExaHyPE 2 Python core, so these must be added to the Python path, which is the list of locations where Python looks for source files.

This can be done in the terminal via `export PYTHONPATH=$PYTHONPATH:$PEANO_ROOT/python`, or directly within the Python code as below.
Once on the Python path, they can be imported just like any other package:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../python/"))
import exahype2

## Defining a Solver

Here, we define an ExaHyPE 2 solver, specifically an ADER-DG solver.
It takes as arguments a name, a polynomial order, and the unknowns.

In [ ]:
unknowns = {"p": 1, "v": 2}
auxiliary_variables = {}
my_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                  = "PlanarAcousticSolver",
    order                 = 5,
    time_step_relaxation  = 0.9,
    unknowns              = unknowns,
    auxiliary_variables   = auxiliary_variables
)

With our solver defined, we can let it know what our equation will look like.
In this case, we will let it know that it only needs a classical flux.
`exahype2.solvers.PDETerms.User_Defined_Implementation` lets the project know that we will manually define the flux in an implementation file.

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue=exahype2.solvers.PDETerms.User_Defined_Implementation,
)

In addition, let's add some kernel optimisations.
The acoustic equations are linear, which lets us use an optimised version of ADER-DG, so let's specify that.

In [ ]:
my_solver.add_kernel_optimisations(is_linear=True)

## Defining an ExaHyPE 2 Project:

Here, we define an ExaHyPE 2 project to which our solver will be bound.
This takes a namespace for the C++ code and a name for the executable.
We then add our solver to this project.

In [ ]:
exahype2_project = exahype2.Project(
    namespace     = ["tutorials", "exahype2", "acoustic"],
    project_name  = "PlanarWaves",
    executable    = "PlanarWaves",
    dimensions    = 2,
    size          = [2.0,  2.0],
    offset        = [-1.0, -1.0],
    periodic_bc   = [True, True],
    min_end_time  = 1.414,
    max_end_time  = 1.414,
    min_grid_levels = 3,
    max_grid_levels = 3,
)

exahype2_project.add_solver(my_solver)

Now that we have an ExaHyPE 2 project, let's populate it with some information:
This is essentially the domain our solver will run in.
The project needs to know the domain size, how long the simulation should run,
how often it should plot the solution, and what type of boundary we use.

In [ ]:
runtime_parameters = [
    "-et", "1.414",
    "-fpt", "0.0",
    "-ns", "14",
    "-o", "solutions",
]


Finally, we configure for parallel runs and compile the project.

In [ ]:
exahype2_project.output.makefile.add_cpp_file("PlanarAcousticSolver.cpp")
exahype2_project.generate_and_build()

Once the implementation has compiled, we can run it.
This is usually done directly in the terminal via `./$executable` but can be done in Python as well.

In [ ]:
import subprocess
subprocess.run(["./PlanarWaves", *runtime_parameters], check=True)
